# 08 · Preprocesamiento y Feature Engineering

En datos tabulares, una gran parte del rendimiento viene de representar correctamente el problema. Este lab cubre missing values, outliers, encoding, scaling, transformaciones, fechas, interacciones y pipelines heterogéneos.

## Objetivos
- Auditar calidad antes de modelar.
- Entender cuándo escalar y cuándo no.
- Imputar sin leakage.
- Codificar variables categóricas.
- Crear features numéricas, temporales e interacciones.
- Usar `ColumnTransformer` para un pipeline reproducible.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler, PowerTransformer, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
SEED=42
rng=np.random.default_rng(SEED); n=1500
df=pd.DataFrame({
 'edad':rng.normal(42,14,n).clip(18,90),
 'ingreso':rng.lognormal(10.5,.7,n),
 'region':rng.choice(['norte','centro','sur'],n,p=[.2,.55,.25]),
 'canal':rng.choice(['web','presencial','telefono'],n),
 'fecha':pd.Timestamp('2025-01-01')+pd.to_timedelta(rng.integers(0,365,n),unit='D')
})
logit=-3+.045*(df.edad-40)+.000015*df.ingreso+(df.region=='centro')*.6+(df.canal=='web')*.5
df['target']=rng.binomial(1,1/(1+np.exp(-logit)))
# introducir missing y outlier
df.loc[rng.choice(n,100,replace=False),'ingreso']=np.nan
df.loc[rng.choice(n,40,replace=False),'region']=None
df.loc[0,'ingreso']=1e8
df.head()

## 1. Auditoría de calidad
Antes de modelar: tipos, duplicados, missing, rangos imposibles, cardinalidad, clases raras y consistencia temporal. Una feature técnicamente válida puede ser semánticamente incorrecta.


In [ ]:
audit=pd.DataFrame({
 'dtype':df.dtypes.astype(str),
 'missing':df.isna().sum(),
 'missing_pct':df.isna().mean(),
 'nunique':df.nunique(dropna=False)
})
display(audit); display(df.describe(include='all',datetime_is_numeric=True) if False else df.describe(include='all'))

## 2. Imputación
- Media: sensible a outliers.
- Mediana: robusta en numéricas sesgadas.
- Moda: baseline categórico.
- Indicador de missing: útil si la ausencia tiene señal.
- KNN/Iterative imputation: más sofisticados, pero deben entrenarse dentro de CV.

A veces `missing` es una categoría legítima, no un error.


## 3. Scaling y transformaciones
`StandardScaler` es importante para regresión regularizada, SVM, KNN, PCA y redes. Árboles casi nunca lo necesitan. `RobustScaler` usa mediana/IQR y tolera outliers. `PowerTransformer` o `log1p` pueden reducir sesgo fuerte.


In [ ]:
fig,ax=plt.subplots(1,2,figsize=(10,4)); df.ingreso.clip(upper=df.ingreso.quantile(.99)).hist(bins=40,ax=ax[0]); ax[0].set_title('ingreso original'); np.log1p(df.ingreso).hist(bins=40,ax=ax[1]); ax[1].set_title('log1p'); plt.show()

## 4. Fechas y variables cíclicas
Una fecha cruda rara vez entra bien a un modelo. Podemos extraer año, mes, día de semana, antigüedad o representar ciclos con seno/coseno. Para día de semana:
$$sin(2\pi d/7),\quad cos(2\pi d/7)$$
Así domingo y lunes quedan cercanos geométricamente.


In [ ]:
work=df.copy(); work['dow']=work.fecha.dt.dayofweek; work['month']=work.fecha.dt.month; work['days_since_start']=(work.fecha-work.fecha.min()).dt.days
work['dow_sin']=np.sin(2*np.pi*work.dow/7); work['dow_cos']=np.cos(2*np.pi*work.dow/7)
work=work.drop(columns=['fecha'])

## 5. ColumnTransformer: pipeline heterogéneo
Ajusta imputación, scaling y one-hot **solo con train** dentro de cada fold. `handle_unknown='ignore'` evita fallar cuando aparece una categoría nueva en inferencia.


In [ ]:
X=work.drop(columns='target'); y=work.target
num=X.select_dtypes(include=np.number).columns.tolist(); cat=X.select_dtypes(exclude=np.number).columns.tolist()
pre=ColumnTransformer([
 ('num',Pipeline([('imputer',SimpleImputer(strategy='median',add_indicator=True)),('scale',RobustScaler())]),num),
 ('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat)
])
pipe=Pipeline([('pre',pre),('model',LogisticRegression(max_iter=4000))])
print('CV ROC-AUC',cross_val_score(pipe,X,y,cv=5,scoring='roc_auc').mean())

## 6. Feature engineering de dominio
Ejemplos:
- razones: gasto/ingreso, uso/capacidad;
- conteos en ventana: eventos últimos 7/30/90 días;
- recency/frequency/monetary;
- interacciones edad×segmento;
- agregaciones por persona/hogar/territorio;
- NLP: TF-IDF/embeddings;
- imagen/audio: embeddings de modelos preentrenados.

**Regla:** una buena feature debe poder calcularse exactamente igual en entrenamiento e inferencia.

## Leakage frecuente
- agregaciones que incluyen datos posteriores al momento de predicción;
- target encoding calculado usando la propia fila;
- estadísticas globales antes del split;
- selección de features usando test.

## Ejercicios
1. Añade `MissingIndicator` y compara.
2. Compara StandardScaler vs RobustScaler.
3. Implementa winsorization de outliers dentro de un transformer custom.
4. Crea target encoding con CV sin leakage.
5. Construye features RFM sintéticas.
6. Crea un test unitario que garantice que el pipeline acepta categorías no vistas.
7. Investiga hashing trick, frequency encoding y learned embeddings categóricos.
